In [1]:
import json
import math

In [2]:
with open("text_segmentation_dataset.json", "r", encoding="utf-8") as file:
    data = json.load(file)

print("Dataset loaded successfully")

Dataset loaded successfully


In [3]:
word_counts = data["word_counts"]
test_cases = data["test_cases"]

print("Vocabulary size:", len(word_counts))
print("Number of test cases:", len(test_cases))

Vocabulary size: 1500
Number of test cases: 1000


In [4]:
print(test_cases[0])

{'input': 'itthatthecitytakestepstothisproblem', 'ground_truth': 'it that the city take steps to this problem', 'word_count': 9}


In [5]:
vocabulary = set(word_counts.keys())

print("Vocabulary size:", len(vocabulary))

Vocabulary size: 1500


In [6]:
max_word_length = 0

for word in vocabulary:
    if len(word) > max_word_length:
        max_word_length = len(word)

print("Maximum word length:", max_word_length)

Maximum word length: 14


In [7]:
def greedy_segment(text):
    words = []
    i = 0

    while i < len(text):

        best_word = None

        max_len = min(max_word_length, len(text) - i)

        # Try longest word first
        for length in range(max_len, 0, -1):

            candidate = text[i:i + length]

            if candidate in vocabulary:
                best_word = candidate
                break

        # If no word is found
        if best_word is None:
            best_word = text[i]

        words.append(best_word)

        i = i + len(best_word)

    return words

In [8]:
example = test_cases[0]["input"]

print("Input:")
print(example)

print("\nGreedy Segmentation:")
print(greedy_segment(example))

Input:
itthatthecitytakestepstothisproblem

Greedy Segmentation:
['it', 'that', 'the', 'city', 'takes', 't', 'e', 'p', 's', 'to', 'this', 'problem']


In [9]:
ground_truth = test_cases[0]["ground_truth"].split()

print("Ground Truth:")
print(ground_truth)

print("\nGreedy Result:")
print(greedy_segment(example))

Ground Truth:
['it', 'that', 'the', 'city', 'take', 'steps', 'to', 'this', 'problem']

Greedy Result:
['it', 'that', 'the', 'city', 'takes', 't', 'e', 'p', 's', 'to', 'this', 'problem']


In [ ]:
#Dynamic Programming Approach

In [10]:
total_frequency = 0

for word in word_counts:
    total_frequency += word_counts[word]

print("Total frequency:", total_frequency)

Total frequency: 735040


In [11]:
def dp_segment(text):

    n = len(text)

    # dp[i] stores the maximum log probability
    # for segmenting text[0:i]
    dp = [-float("inf")] * (n + 1)

    # best_word[i] stores the last word
    # used to reach position i
    best_word = [None] * (n + 1)

    # Empty text has score 0
    dp[0] = 0.0

    for i in range(n):

        if dp[i] == -float("inf"):
            continue

        max_len = min(max_word_length, n - i)

        for length in range(1, max_len + 1):

            word = text[i:i + length]

            if word in word_counts:

                frequency = word_counts[word]

                probability = frequency / total_frequency

                log_probability = math.log(probability)

                new_score = dp[i] + log_probability

                if new_score > dp[i + length]:

                    dp[i + length] = new_score

                    best_word[i + length] = word

    # If no complete segmentation was found
    if best_word[n] is None:
        return greedy_segment(text)

    # Reconstruct words
    words = []

    position = n

    while position > 0:

        word = best_word[position]

        if word is None:
            return greedy_segment(text)

        words.append(word)

        position = position - len(word)

    # Reverse because reconstruction happens backwards
    words.reverse()

    return words

In [12]:
print("Input:")
print(example)

print("\nDynamic Programming:")
print(dp_segment(example))

Input:
itthatthecitytakestepstothisproblem

Dynamic Programming:
['it', 'that', 'the', 'city', 'take', 'steps', 'to', 'this', 'problem']


In [13]:
ground_truth = test_cases[0]["ground_truth"].split()

greedy_result = greedy_segment(example)

dp_result = dp_segment(example)

print("Ground Truth:")
print(ground_truth)

print("\nGreedy:")
print(greedy_result)

print("\nDynamic Programming:")
print(dp_result)

Ground Truth:
['it', 'that', 'the', 'city', 'take', 'steps', 'to', 'this', 'problem']

Greedy:
['it', 'that', 'the', 'city', 'takes', 't', 'e', 'p', 's', 'to', 'this', 'problem']

Dynamic Programming:
['it', 'that', 'the', 'city', 'take', 'steps', 'to', 'this', 'problem']


In [ ]:
#Edit Distance

In [14]:
def edit_distance(predicted, actual):

    m = len(predicted)
    n = len(actual)

    # Create DP table
    dp = [[0] * (n + 1) for _ in range(m + 1)]

    # First column
    for i in range(m + 1):
        dp[i][0] = i

    # First row
    for j in range(n + 1):
        dp[0][j] = j

    # Fill the table
    for i in range(1, m + 1):

        for j in range(1, n + 1):

            if predicted[i - 1] == actual[j - 1]:
                cost = 0
            else:
                cost = 1

            insertion = dp[i][j - 1] + 1

            deletion = dp[i - 1][j] + 1

            substitution = dp[i - 1][j - 1] + cost

            dp[i][j] = min(
                insertion,
                deletion,
                substitution
            )

    return dp[m][n]

In [15]:
actual = ["the", "cat", "is", "here"]

predicted = ["the", "cat", "here"]

distance = edit_distance(predicted, actual)

print("Actual:", actual)
print("Predicted:", predicted)
print("Edit Distance:", distance)

Actual: ['the', 'cat', 'is', 'here']
Predicted: ['the', 'cat', 'here']
Edit Distance: 1


In [ ]:
# accuracy 

In [16]:
def evaluate_method(segment_function):

    correct = 0

    total_edit_distance = 0

    total_cases = len(test_cases)

    for case in test_cases:

        text = case["input"]

        actual = case["ground_truth"].split()

        predicted = segment_function(text)

        # Accuracy
        if predicted == actual:
            correct += 1

        # Edit Distance
        distance = edit_distance(predicted, actual)

        total_edit_distance += distance

    accuracy = (correct / total_cases) * 100

    average_edit_distance = total_edit_distance / total_cases

    return accuracy, average_edit_distance

In [17]:
greedy_accuracy, greedy_edit_distance = evaluate_method(
    greedy_segment
)

print("Greedy Longest Match")
print("-------------------------")
print("Accuracy:", greedy_accuracy, "%")
print("Average Edit Distance:", greedy_edit_distance)

Greedy Longest Match
-------------------------
Accuracy: 69.1 %
Average Edit Distance: 1.26


In [18]:
dp_accuracy, dp_edit_distance = evaluate_method(
    dp_segment
)

print("Dynamic Programming")
print("-------------------------")
print("Accuracy:", dp_accuracy, "%")
print("Average Edit Distance:", dp_edit_distance)

Dynamic Programming
-------------------------
Accuracy: 98.2 %
Average Edit Distance: 0.037


In [19]:
print("=" * 60)
print("TEXT SEGMENTATION PERFORMANCE")
print("=" * 60)

print()

print("Method                    Accuracy       Edit Distance")
print("-" * 60)

print(
    "Greedy Longest Match      {:.2f}%          {:.3f}".format(
        greedy_accuracy,
        greedy_edit_distance
    )
)

print(
    "Dynamic Programming       {:.2f}%          {:.3f}".format(
        dp_accuracy,
        dp_edit_distance
    )
)

print("=" * 60)

TEXT SEGMENTATION PERFORMANCE

Method                    Accuracy       Edit Distance
------------------------------------------------------------
Greedy Longest Match      69.10%          1.260
Dynamic Programming       98.20%          0.037


In [20]:
print("RESULT")
print("=" * 50)

if dp_accuracy > greedy_accuracy:
    print("Dynamic Programming has better Accuracy.")
elif greedy_accuracy > dp_accuracy:
    print("Greedy has better Accuracy.")
else:
    print("Both methods have the same Accuracy.")

if dp_edit_distance < greedy_edit_distance:
    print("Dynamic Programming has lower Edit Distance.")
elif greedy_edit_distance < dp_edit_distance:
    print("Greedy has lower Edit Distance.")
else:
    print("Both methods have the same Edit Distance.")

RESULT
Dynamic Programming has better Accuracy.
Dynamic Programming has lower Edit Distance.


In [21]:
for i in range(min(10, len(test_cases))):

    case = test_cases[i]

    text = case["input"]

    actual = case["ground_truth"].split()

    greedy_result = greedy_segment(text)

    dp_result = dp_segment(text)

    print("=" * 70)

    print("Test Case:", i + 1)

    print("\nInput:")
    print(text)

    print("\nGround Truth:")
    print(actual)

    print("\nGreedy:")
    print(greedy_result)

    print("\nDynamic Programming:")
    print(dp_result)

Test Case: 1

Input:
itthatthecitytakestepstothisproblem

Ground Truth:
['it', 'that', 'the', 'city', 'take', 'steps', 'to', 'this', 'problem']

Greedy:
['it', 'that', 'the', 'city', 'takes', 't', 'e', 'p', 's', 'to', 'this', 'problem']

Dynamic Programming:
['it', 'that', 'the', 'city', 'take', 'steps', 'to', 'this', 'problem']
Test Case: 2

Input:
oftitlelawwasalsobythe

Ground Truth:
['of', 'title', 'law', 'was', 'also', 'by', 'the']

Greedy:
['of', 'title', 'law', 'was', 'also', 'by', 'the']

Dynamic Programming:
['of', 'title', 'law', 'was', 'also', 'by', 'the']
Test Case: 3

Input:
failuretodothiswillcontinuetoplaceaon

Ground Truth:
['failure', 'to', 'do', 'this', 'will', 'continue', 'to', 'place', 'a', 'on']

Greedy:
['failure', 'to', 'do', 'this', 'will', 'continue', 'top', 'l', 'a', 'c', 'e', 'a', 'on']

Dynamic Programming:
['failure', 'to', 'do', 'this', 'will', 'continue', 'to', 'place', 'a', 'on']
Test Case: 4

Input:
onotherthethat

Ground Truth:
['on', 'other', 'the', '

In [22]:
count = 0

for case in test_cases:

    text = case["input"]

    greedy_result = greedy_segment(text)

    dp_result = dp_segment(text)

    if greedy_result != dp_result:

        print("=" * 70)

        print("Input:")
        print(text)

        print("\nGround Truth:")
        print(case["ground_truth"].split())

        print("\nGreedy:")
        print(greedy_result)

        print("\nDynamic Programming:")
        print(dp_result)

        count += 1

        if count == 10:
            break

Input:
itthatthecitytakestepstothisproblem

Ground Truth:
['it', 'that', 'the', 'city', 'take', 'steps', 'to', 'this', 'problem']

Greedy:
['it', 'that', 'the', 'city', 'takes', 't', 'e', 'p', 's', 'to', 'this', 'problem']

Dynamic Programming:
['it', 'that', 'the', 'city', 'take', 'steps', 'to', 'this', 'problem']
Input:
failuretodothiswillcontinuetoplaceaon

Ground Truth:
['failure', 'to', 'do', 'this', 'will', 'continue', 'to', 'place', 'a', 'on']

Greedy:
['failure', 'to', 'do', 'this', 'will', 'continue', 'top', 'l', 'a', 'c', 'e', 'a', 'on']

Dynamic Programming:
['failure', 'to', 'do', 'this', 'will', 'continue', 'to', 'place', 'a', 'on']
Input:
forthesaidthatanpropertyhasbeenagreedupon

Ground Truth:
['for', 'the', 'said', 'that', 'an', 'property', 'has', 'been', 'agreed', 'upon']

Greedy:
['forth', 'e', 'said', 'that', 'an', 'property', 'has', 'been', 'agreed', 'upon']

Dynamic Programming:
['for', 'the', 'said', 'that', 'an', 'property', 'has', 'been', 'agreed', 'upon']
Input